In [20]:
# 1. Import Packages
import pandas as pd
import numpy as np
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
# Import the Logistic Regression model and evaluation metrics from scikit-learn
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

In [21]:
#read data
credit_df = pd.read_csv("https://raw.githubusercontent.com/ammishra08/Machine-Learning/refs/heads/master/Datasets/credit_train.csv", header=0, sep=',')
credit_df.head()

,Loan ID,Customer ID,Loan Status,Current Loan Amount,Term,Credit Score,Annual Income,Years in current job,Home Ownership,Purpose,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
0,14dd8831-6af5-400b-83ec-68e61888a048,981165ec-3274-42f5-a3b4-d104041a9ca9,Fully Paid,445412,Short Term,709.0,1167493.0,8 years,Home Mortgage,Home Improvements,5214.74,17.2,NaN,6,1,228190,416746.0,1.0,0.0
1,4771cc26-131a-45db-b5aa-537ea4ba5342,2de017a3-2e01-49cb-a581-08169e83be29,Fully Paid,262328,Short Term,NaN,NaN,10+ years,Home Mortgage,Debt Consolidation,33295.98,21.1,8.0,35,0,229976,850784.0,0.0,0.0
2,4eed4e6a-aa2f-4c91-8651-ce984ee8fb26,5efb2b2b-bf11-4dfd-a572-3761a2694725,Fully Paid,99999999,Short Term,741.0,2231892.0,8 years,Own Home,Debt Consolidation,29200.53,14.9,29.0,18,1,297996,750090.0,0.0,0.0
3,77598f7b-32e7-4e3b-a6e5-06ba0d98fe8a,e777faab-98ae-45af-9a86-7ce5b33b1011,Fully Paid,347666,Long Term,721.0,806949.0,3 years,Own Home,Debt Consolidation,8741.90,12.0,NaN,9,0,256329,386958.0,0.0,0.0
4,d4062e70-befa-4995-8643-a0de73938182,81536ad9-5ccf-4eb8-befb-47a4d608658e,Fully Paid,176220,Short Term,NaN,NaN,5 years,Rent,Debt Consolidation,20639.70,6.1,NaN,15,0,253460,427174.0,0.0,0.0


In [22]:
# 3. Data Processing & Cleansing
credit_df['Months since last delinquent'] = credit_df['Months since last delinquent'].fillna(0)
 
credit_df['Credit Score'] = credit_df[credit_df['Credit Score'] >= 900]['Credit Score'] / 10
 
# Fill missing credit score value by average of credit score
credit_df['Credit Score'] = credit_df['Credit Score'].fillna(round(credit_df['Credit Score'].mean()))
 
# Fill missing Annual Income by average of mean of Annual Income
credit_df['Annual Income'] = credit_df['Annual Income'].fillna(round(credit_df['Annual Income'].mean()))
 

In [23]:
credit_df['Years in current job'] = credit_df['Years in current job'].str.replace(' years', '').str.replace(' year', '').str.replace('< 1', '0.5').str.replace('+', '')

In [24]:
# Clean the text first
credit_df['Years in current job'] = (
    credit_df['Years in current job']
    .astype(str)
    .str.replace(' years', '', regex=False)
    .str.replace(' year', '', regex=False)
    .str.replace('< 1', '0.5', regex=False)
    .str.replace('+', '', regex=False)
    .str.strip()
)

# Convert to numeric; invalid values become NaN
credit_df['Years in current job'] = pd.to_numeric(
    credit_df['Years in current job'],
    errors='coerce'
)

# Fill missing values with median
credit_df['Years in current job'] = credit_df['Years in current job'].fillna(
    credit_df['Years in current job'].median()
)
 

In [25]:
credit_df.head()




,Loan ID,Customer ID,Loan Status,Current Loan Amount,Term,Credit Score,Annual Income,Years in current job,Home Ownership,Purpose,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
0,14dd8831-6af5-400b-83ec-68e61888a048,981165ec-3274-42f5-a3b4-d104041a9ca9,Fully Paid,445412,Short Term,711.0,1167493.0,8.0,Home Mortgage,Home Improvements,5214.74,17.2,0.0,6,1,228190,416746.0,1.0,0.0
1,4771cc26-131a-45db-b5aa-537ea4ba5342,2de017a3-2e01-49cb-a581-08169e83be29,Fully Paid,262328,Short Term,711.0,1378277.0,10.0,Home Mortgage,Debt Consolidation,33295.98,21.1,8.0,35,0,229976,850784.0,0.0,0.0
2,4eed4e6a-aa2f-4c91-8651-ce984ee8fb26,5efb2b2b-bf11-4dfd-a572-3761a2694725,Fully Paid,99999999,Short Term,711.0,2231892.0,8.0,Own Home,Debt Consolidation,29200.53,14.9,29.0,18,1,297996,750090.0,0.0,0.0
3,77598f7b-32e7-4e3b-a6e5-06ba0d98fe8a,e777faab-98ae-45af-9a86-7ce5b33b1011,Fully Paid,347666,Long Term,711.0,806949.0,3.0,Own Home,Debt Consolidation,8741.90,12.0,0.0,9,0,256329,386958.0,0.0,0.0
4,d4062e70-befa-4995-8643-a0de73938182,81536ad9-5ccf-4eb8-befb-47a4d608658e,Fully Paid,176220,Short Term,711.0,1378277.0,5.0,Rent,Debt Consolidation,20639.70,6.1,0.0,15,0,253460,427174.0,0.0,0.0


In [26]:

credit_df.describe()

,Current Loan Amount,Credit Score,Annual Income,Years in current job,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
count,1.000000e+05,100000.000000,1.000000e+05,100000.000000,100000.000000,100000.000000,100000.000000,100000.00000,100000.000000,1.000000e+05,9.999800e+04,99796.000000,99990.000000
mean,1.176045e+07,710.995400,1.378277e+06,5.937720,18472.412336,18.199141,16.354410,11.12853,0.168310,2.946374e+05,7.607984e+05,0.117740,0.029313
std,3.178394e+07,6.540649,9.722974e+05,3.487901,12174.992609,7.015324,23.023402,5.00987,0.482705,3.761709e+05,8.384503e+06,0.351424,0.258182
min,1.080200e+04,585.000000,7.662700e+04,0.500000,0.000000,3.600000,0.000000,0.00000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000
25%,1.796520e+05,711.000000,9.311805e+05,3.000000,10214.162500,13.500000,0.000000,8.00000,0.000000,1.126700e+05,2.734380e+05,0.000000,0.000000
50%,3.122460e+05,711.000000,1.369919e+06,6.000000,16220.300000,16.900000,0.000000,10.00000,0.000000,2.098170e+05,4.678740e+05,0.000000,0.000000
75%,5.249420e+05,711.000000,1.512704e+06,10.000000,24012.057500,21.700000,29.000000,14.00000,0.000000,3.679588e+05,7.829580e+05,0.000000,0.000000
max,1.000000e+08,751.000000,1.655574e+08,10.000000,435843.280000,70.500000,176.000000,76.00000,15.000000,3.287897e+07,1.539738e+09,7.000000,15.000000


In [27]:
# Drop unnecessary columns
for col in ['Loan ID', 'Customer ID']:
    if col in credit_df.columns:
        credit_df = credit_df.drop(columns=[col])

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Find categorical columns
categorical_cols = credit_df.select_dtypes(include=['object']).columns

print("Categorical Columns:", list(categorical_cols))

# Apply Label Encoding
for col in categorical_cols:
    credit_df[col] = le.fit_transform(credit_df[col].astype(str))

# Check result
print(credit_df.head())

Categorical Columns: ['Loan Status', 'Term', 'Home Ownership', 'Purpose']
   Loan Status  Current Loan Amount  Term  Credit Score  Annual Income  \
0            1               445412     1         711.0      1167493.0   
1            1               262328     1         711.0      1378277.0   
2            1             99999999     1         711.0      2231892.0   
3            1               347666     0         711.0       806949.0   
4            1               176220     1         711.0      1378277.0   

   Years in current job  Home Ownership  Purpose  Monthly Debt  \
0                   8.0               1        5       5214.74   
1                  10.0               1        3      33295.98   
2                   8.0               2        3      29200.53   
3                   3.0               2        3       8741.90   
4                   5.0               3        3      20639.70   

   Years of Credit History  Months since last delinquent  \
0                     17

In [29]:
# 6. Features & Target
X = credit_df.drop('Loan Status', axis = 1)
Y = credit_df['Loan Status']
 

In [30]:
X.head()

,Current Loan Amount,Term,Credit Score,Annual Income,Years in current job,Home Ownership,Purpose,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
0,445412,1,711.0,1167493.0,8.0,1,5,5214.74,17.2,0.0,6,1,228190,416746.0,1.0,0.0
1,262328,1,711.0,1378277.0,10.0,1,3,33295.98,21.1,8.0,35,0,229976,850784.0,0.0,0.0
2,99999999,1,711.0,2231892.0,8.0,2,3,29200.53,14.9,29.0,18,1,297996,750090.0,0.0,0.0
3,347666,0,711.0,806949.0,3.0,2,3,8741.90,12.0,0.0,9,0,256329,386958.0,0.0,0.0
4,176220,1,711.0,1378277.0,5.0,3,3,20639.70,6.1,0.0,15,0,253460,427174.0,0.0,0.0


In [31]:
Y.head()

0    1
1    1
2    1
3    1
4    1
Name: Loan Status, dtype: int64

In [ ]:
# # 1. Import Packages
# import pandas as pd
# import numpy as np
# # Split the data into training and testing sets
# from sklearn.model_selection import train_test_split
# # Import the Logistic Regression model and evaluation metrics from scikit-learn
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
# import warnings
# warnings.filterwarnings('ignore')
 
# # 2. Read Data
# credit_df = pd.read_csv("https://raw.githubusercontent.com/ammishra08/Machine-Learning/refs/heads/master/Datasets/credit_train.csv", header=0, sep=',')

 
# # 3. Data Processing & Cleansing
# credit_df['Months since last delinquent'] = credit_df['Months since last delinquent'].fillna(0)
 
# credit_df['Credit Score'] = credit_df[credit_df['Credit Score'] >= 900]['Credit Score'] / 10
 
# # Fill missing credit score value by average of credit score
# credit_df['Credit Score'] = credit_df['Credit Score'].fillna(round(credit_df['Credit Score'].mean()))
 
# # Fill missing Annual Income by average of mean of Annual Income
# credit_df['Annual Income'] = credit_df['Annual Income'].fillna(round(credit_df['Annual Income'].mean()))
 
# credit_df['Years in current job'] = credit_df['Years in current job'].str.replace(' years', '').str.replace(' year', '').str.replace('< 1', '0.5').str.replace('+', '')
# credit_df['Years in current job'] = credit_df['Years in current job'].astype('float')
 
# credit_df.fillna({'Years in current job': credit_df['Years in current job'].median()}, inplace = True)
 
# credit_df.dropna(inplace = True)
 
# print(credit_df.head())
# print(credit_df.describe())
 
# # 4. Drop Columns - 'Loan ID' & 'Customer ID'
# credit_df.drop(['Loan ID', 'Customer ID'], axis = 1, inplace=True)
 
# # 5. Data Pre-Processing - Handling All Categorical Values
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
 
# binary_cols = [col for col in credit_df.columns if credit_df[col].dtype == 'str' and
#                credit_df[col].nunique()>1]
# print(binary_cols)
 
# for col in binary_cols:
#     # fit() - train label encoder with input samples
#     # transform() - apply changes and transform existing dataframe samples.
#     credit_df[col] = le.fit_transform(credit_df[col])
 
# # 6. Features & Target
# X = credit_df.drop('Loan Status', axis = 1)
# Y = credit_df['Loan Status']
 
# print(X.head())
# print(Y.head())
 
 

In [ ]:
# X.head()

In [ ]:
# Y.head()

In [ ]:
# !pip install imbalanced-learn==0.14.2
#


   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -

In [34]:
# Add Smote

from imblearn.over_sampling import SMOTE
smote = SMOTE()
# Handle missing values before oversampling

# If there are NaNs anywhere in X, fill them before SMOTE
if X.isnull().any().any():
    imputer = SimpleImputer(strategy="median")
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Apply SMOTE after imputing missing values
transformed_feature, transformed_label = smote.fit_resample(X, Y)

In [35]:
# 9. Split Data into Train & Test
X_train, X_test, Y_train, Y_test = train_test_split(transformed_feature, transformed_label, test_size = 0.2,
                                                    random_state = 2)

In [39]:
# 10. Decision Tree Classifier
clf_tree_best = DecisionTreeClassifier(ccp_alpha = 0.001, criterion = 'gini',
                                       max_depth = 20, random_state = 2)
 
clf_tree_best.fit(X_train, Y_train)
 
joblib.dump(clf_tree_best, "../model/loan_default.pkl")
 
print("\nModel Saved Successfully!")
 


Model Saved Successfully!


In [43]:
# save test sample as csv for prediction
X_test.to_csv("../data/x_test_sample.csv")